# 00 · Data basics: xarray → NumPy → torch

Goal: learn the syntax you need for `dataset.py` on our real data, one small step at a time.

How to use this notebook:
- Run the **example** cells and read the output.
- In each **✏️ exercise**, write down your *prediction* (shape / number) in a comment **before** running.
- Exercises use *different* years/variables from `load_split`, so the function itself is still yours to write.

Docs used here:
- xarray: [Data structures](https://docs.xarray.dev/en/stable/user-guide/data-structures.html) ·
  [Indexing and selecting data](https://docs.xarray.dev/en/stable/user-guide/indexing.html) ·
  [Plotting](https://docs.xarray.dev/en/stable/user-guide/plotting.html)
- PyTorch: [Tensors tutorial](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html), section *Bridge with NumPy*

## 0 · Setup

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import torch
import matplotlib.pyplot as plt

# find the repo root (the folder that contains data/), wherever this notebook is run from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
PATH = ROOT / "data" / "era5_ireland_wind_daily_max.nc"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PATH

## 1 · Opening the file: the `Dataset`

`xr.open_dataset` is **lazy**: it reads names, dimensions and coordinates, not the numbers.
A `Dataset` is like a dict of labelled arrays that share coordinates.

In [ ]:
ds = xr.open_dataset(PATH)
ds

In [ ]:
print("dimensions:", dict(ds.sizes))
print("variables :", list(ds.data_vars))
print("coords    :", list(ds.coords))

✏️ **Exercise 1.** How many grid cells does one day have? How many numbers are stored for one variable
over all days? Predict, then compute both from `ds.sizes`.

In [ ]:
# your code here

## 2 · One variable: the `DataArray`

`ds["gust"]` picks one variable. A `DataArray` = a NumPy array **+ names for each axis + coordinate labels + attributes**.

In [ ]:
gust = ds["gust"]
print("dims :", gust.dims)
print("shape:", gust.shape)
print("dtype:", gust.dtype)
print("attrs:", gust.attrs)
gust

In [ ]:
# coordinates are themselves small DataArrays
print(gust.latitude.values[:3], "...", gust.latitude.values[-1])
print(gust.longitude.values[:3], "...", gust.longitude.values[-1])
print(gust.time.values[:2])

✏️ **Exercise 2.** Pick the `speed` variable instead. Predict its `dims` and `shape`, then check.
Is `latitude` increasing or decreasing? (This matters for plotting with `imshow` later.)

In [ ]:
# your code here

## 3 · Selecting data: `.isel` (by position) vs `.sel` (by label)

- `.isel(time=0)`: the first day, like `array[0]`.
- `.sel(time="2015-01-08")`: the day with that **label**. Safer: no counting rows.
- `slice(a, b)` in `.sel` **includes both ends** (unlike Python's `a:b`).
- A partial date string like `"1990"` or `"1990-02"` means "the whole year / month".

In [ ]:
first_day = gust.isel(time=0)
storm_day = gust.sel(time="2015-01-08")
print("first day :", first_day.shape, str(first_day.time.values)[:10])
print("storm day :", storm_day.shape, f"max {float(storm_day.max()):.1f} m/s")

In [ ]:
feb_1990 = gust.sel(time="1990-02")                       # one month
nineties = gust.sel(time=slice("1990", "1999"))           # a range of years, both ends included
print("Feb 1990:", feb_1990.shape)
print("1990s   :", nineties.shape)

In [ ]:
# a single grid point: Dublin is ~53.35N, 6.26W; method="nearest" snaps to the closest cell
dublin = gust.sel(latitude=53.35, longitude=-6.26, method="nearest")
print(dublin.shape, "at", float(dublin.latitude), float(dublin.longitude))

✏️ **Exercise 3a.** Select `speed` for **2016** only. Predict the number of days first (careful: leap year).

✏️ **Exercise 3b.** Select `gust` for the years **2000 to 2004**. Predict the shape first.
Then do the same with the years stored as integers, e.g. `years = (2000, 2004)`. What do you have to
do to the integers before `slice` works with them?

In [ ]:
# 3a

In [ ]:
# 3b

## 4 · Reductions: collapse a dimension by name

You say *which* dimension to reduce by name, not by axis number. That's the main advantage over NumPy.

In [ ]:
clim = gust.mean("time")                                  # average map over all days -> (32, 32)
domain_max = gust.max(["latitude", "longitude"])          # windiest pixel each day -> (14610,)
print("climatology:", clim.shape, "| domain max per day:", domain_max.shape)

windiest = domain_max.idxmax("time")                      # the label (date) of the maximum
print("windiest day:", str(windiest.values)[:10], f"{float(domain_max.max()):.1f} m/s")

In [ ]:
monthly = domain_max.groupby("time.month").mean()         # seasonal cycle
monthly.round(1).to_series()

✏️ **Exercise 4.** Using `speed`: (a) compute the mean map over 2015–2019 only, and predict its shape.
(b) find the 3 windiest days by domain max. Hint: `.sortby(...)` or `np.argsort` on `.values`.

In [ ]:
# your code here

## 5 · Looking at it: plots

`imshow` puts row 0 at the **top** by default. Our latitude runs south → north (row 0 = 49.75°N),
so use `origin="lower"`; otherwise Ireland is upside down. `extent` puts real lon/lat on the axes.
Note: cells are ~28 km N–S but ~16 km E–W, so the map is stretched if the pixels are drawn square.
`aspect` fixes that.

In [ ]:
lon, lat = gust.longitude.values, gust.latitude.values
extent = [lon.min() - 0.125, lon.max() + 0.125, lat.min() - 0.125, lat.max() + 0.125]
aspect = 1 / np.cos(np.deg2rad(53.5))                     # makes 1 deg lon look as wide as it really is

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), constrained_layout=True)
for ax, (title, field) in zip(axes, [("Mean daily-max gust 1985–2024", clim), ("8 Jan 2015 (windiest day)", storm_day)]):
    im = ax.imshow(field.values, origin="lower", extent=extent, aspect=aspect, cmap="Blues")
    ax.set(title=title, xlabel="longitude (°E)", ylabel="latitude (°N)"); ax.grid(False)
    fig.colorbar(im, ax=ax, label="gust (m/s)", shrink=0.85)

In [ ]:
# the same map with xarray's built-in plotting: labels and colorbar come from the coords and attrs
storm_day.plot(cmap="Blues", figsize=(5, 4.5));

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
dublin.sel(time=slice("2014-11", "2015-02")).plot(ax=ax, lw=1.5, color="#2a6fb0")
ax.set(title="Daily-max gust at the Dublin grid cell, winter 2014/15", ylabel="gust (m/s)", xlabel="");

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.hist(domain_max.values, bins=60, color="#2a6fb0", edgecolor="white", linewidth=0.5)
ax.set(title="Domain-max gust per day, 1985–2024", xlabel="gust (m/s)", ylabel="number of days");

✏️ **Exercise 5.** Plot the `speed` map for the windiest *speed* day you found in Exercise 4, with
`origin="lower"`. Then plot it again **without** `origin="lower"` and describe what changed.

In [ ]:
# your code here

## 6 · From xarray to NumPy to torch

- `.values` → a NumPy array (the labels are gone, only numbers and shape remain).
- `torch.from_numpy(a)` → a tensor that **shares memory** with `a` (no copy, so it's fast).
- `torch.tensor(a)` → always **copies**.

In [ ]:
a = gust.sel(time="2015-01").values
print(type(a), a.shape, a.dtype)

t = torch.from_numpy(a)
print(type(t), t.shape, t.dtype)

In [ ]:
# shared memory: changing the array changes the tensor too
a_small = np.zeros(3, dtype=np.float32)
t_shared, t_copy = torch.from_numpy(a_small), torch.tensor(a_small)
a_small[0] = 99
print("from_numpy:", t_shared, "| tensor():", t_copy)

In [ ]:
# dtype matters: NumPy defaults to float64, PyTorch layers default to float32
b = np.ones((2, 2))                                       # float64!
print(torch.from_numpy(b).dtype, "->", torch.from_numpy(b).float().dtype)

### Indexing and adding a channel dimension

`Conv2d` expects `(batch, channels, height, width)`. Our maps have no channel axis yet.

In [ ]:
print("t        :", t.shape)            # (31, 32, 32): a month of maps
print("t[0]     :", t[0].shape)         # one map
print("t[0:1]   :", t[0:1].shape)       # slicing keeps the dimension
print("t[0][None]:", t[0][None].shape)  # None inserts a new axis of size 1

✏️ **Exercise 6.** Predict each shape, *then* run:
1. `t.unsqueeze(1)`
2. `t[5].unsqueeze(0)`
3. `torch.stack([t[0], t[1], t[2]])`
4. `torch.stack([t[0].unsqueeze(0), t[1].unsqueeze(0)])`

Which one looks like what a DataLoader would hand to `Conv2d` for a batch of 2 days?

In [ ]:
# your code here

## 7 · Quick tensor statistics

The kind of operation you'll need for normalisation. `dim=` names which axis gets reduced (by number this time).

In [ ]:
print("one number over everything:", t.mean().item())
print("per-day means             :", t.mean(dim=(1, 2)).shape)
print("per-pixel means over days :", t.mean(dim=0).shape)

✏️ **Exercise 7.** For `t` (January 2015): what shape do you get from `t.std(dim=0)`? What does each
number in it mean physically? If you subtract `t.mean(dim=0)` from `t`, why does that work even
though the shapes differ? Look up **broadcasting** in the PyTorch docs
([Broadcasting semantics](https://docs.pytorch.org/docs/stable/notes/broadcasting.html)).

In [ ]:
# your code here

## ✅ You now have every piece of syntax `load_split` needs

Go back to `dataset.py`. Before writing, write the four steps of `load_split` as comments, and next
to each one the line of syntax from this notebook that does it.